CaptchaSolve30k — RF vs GS-Fusion against an evasion ladder of synthetic bots

**Humans:** real sessions from [CaptchaSolve30k](https://huggingface.co/datasets/Capycap-AI/CaptchaSolve30k) (Apache-2.0; 200×200 grid, 240 Hz physics ticks, three puzzle games). Mouse sessions only by default.

**Bots:** generated here, on the **same task skeletons** — each bot session reproduces the waypoints, presses and releases of a real human session (so the task content is identical) and differs only in *how* it moves between them:

| Level | Bot | Movement between waypoints |
|---|---|---|
| L1 | `linear` | straight lines, constant speed, fixed pauses |
| L2 | `linear_jitter` | straight lines, random speed per segment, position noise, random pauses |
| L3 | `bezier_fitts` | curved Bézier paths, Fitts-law duration, ease-in-out speed |
| L4 | `bezier_tremor` | L3 + overshoot and correction, 8–12 Hz tremor, two-phase sub-movements |
| L5 | `human_replay` | **real human segments** from a held-out pool, matched by distance and angle (KD-tree) and rotated/scaled onto each move, with their original timing |
| L5b | `human_replay_raw` | the same, but segments are only moved, **never rotated or scaled** — no warping trace at all, only the joins between segments |

L5 follows the "human replay" evasion level of arXiv 2607.26935. The replay pool is a separate 20 % of humans that are never used for training or testing.

**Models:** logistic regression and Random Forest on 26 movement-dynamics features, and GS-Fusion with three views adapted to mouse data: *statistical* (the same 26 features), *sequence* (LSTM over (dx, dy, button) at 60 Hz), *transition graph* (8-direction Markov transition matrix + grid-cell transition statistics).

**Experiments**
- **E1 — known attackers:** bots of all five levels in training and test.
- **E2 — unseen attacker:** training sees L1–L4 only; both replay variants appear only in the test set. This is the realistic case: defenders train on yesterday's bots.

With `MATCH_SENSOR`, L1–L4 copy the source human's input update rate and coordinate lattice, so they are not caught by sampling artefacts alone (replay segments come from many people, so with `REPLAY_RESAMPLE` the stitched path is rebuilt in continuous time and re-sampled at the source human's update rate and lattice — one device per session, as for a real browser replay bot; see 9c/9d); section 7b reports the best single-feature AUC per level as a check.

Diagnostics after the main run: **9b** which features let RF (trained without replay) catch replay, **9c** whether the replay signal sits at the joins or comes from lattice snapping, **9d** self-replay (a human's own segments) vs cross-human replay, **9e** whether the remaining sub-movement signal sits at the joins.

Metric: recall per level at a threshold that keeps **FPR ≤ 5 %** on validation humans, plus ROC-AUC. Runtime → T4 GPU, or run it in a local Jupyter on your own GPU (results go to ./captcha_results).

In [ ]:
# 1. Setup
%pip install -q datasets lightgbm
import os, math, time, random, warnings, base64, struct, copy
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try: display
except NameError: display = print
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu": print("⚠️  No GPU: Runtime → Change runtime type → T4 GPU, otherwise the neural models take hours.")

In [ ]:
# 2. Configuration
N_PER_GAME    = 2000          # human sessions per game (3 games → 6,000); raise for the final run
INCLUDE_TOUCH = False         # touchscreen sessions behave differently; keep them out by default
MIN_TICKS     = 240           # at least 1 s of input
TICK          = 1 / 240       # seconds per physics tick
BOT_SHARE     = 0.25          # bot sessions per level = BOT_SHARE × usable humans (5 levels → 1.25× humans)
FAST          = True          # True: 30 Hz sequences of ≤ 256 steps, fewer epochs (≈ 4× faster); False for the final run
SEQ_STRIDE    = 8 if FAST else 4      # 240 Hz → 30 / 60 Hz for the sequence view
SEQ_MAX       = 256 if FAST else 512  # max sequence steps (≈ 8.5 s)
SEEDS         = [0, 1, 2, 3, 4]
TARGET_FPR    = 0.05
D, EPOCHS, PATIENCE, BATCH, LR = (32, 25, 4, 512, 2e-3) if FAST else (32, 40, 6, 256, 1e-3)
LAMBDA_BAL    = 0.1
REPLAY_RESAMPLE = True        # replay: rebuild the stitched path in continuous time, then re-sample it at the SOURCE human's
                              # update rate and lattice (one device per session, as a real browser replay bot would have)
MATCH_SENSOR  = True          # L1–L4 copy the source human's input update rate and coordinate lattice (replay is left as recorded)
SKIP_SEQ      = None          # None = automatic: skip the LSTM-based models when there is no GPU (they take hours on CPU)
BASE = "/content" if os.path.isdir("/content") else os.path.abspath(".")   # Colab or a local Jupyter
import datetime
RUN_DATE = datetime.date.today().isoformat()   # every result folder, file and zip gets this suffix, e.g. _2026-11-12
def date_files(folder=None):
    """Add _RUN_DATE to every result file in the folder (progress .pkl files keep their name so a run can resume)."""
    folder = folder or OUT
    for p in os.listdir(folder):
        root, ext = os.path.splitext(p)
        if ext == ".pkl" or root.endswith(RUN_DATE): continue
        os.replace(os.path.join(folder, p), os.path.join(folder, f"{root}_{RUN_DATE}{ext}"))
OUT = f"{BASE}/captcha_results_{RUN_DATE}"; os.makedirs(OUT, exist_ok=True)
LEVELS = ["linear", "linear_jitter", "bezier_fitts", "bezier_tremor", "human_replay", "human_replay_raw"]
REPLAY = ["human_replay", "human_replay_raw"]          # both are hidden from training in E2
GAMES = ["sheep-herding", "thread-the-needle", "polygon-stacking"]
rng = np.random.default_rng(2026); random.seed(2026)

In [ ]:
# 3. Load human sessions (streamed — the full file is 5.3 GB)
from datasets import load_dataset
def stream():
    # the repo has held both data.jsonl and train.jsonl; read one file so sessions are not duplicated
    for f in ["train.jsonl", "data.jsonl"]:
        try:
            ds = load_dataset("Capycap-AI/CaptchaSolve30k", data_files=f, split="train", streaming=True)
            next(iter(ds)); print("reading", f); return ds
        except Exception as e:
            print(f"{f}: not usable ({type(e).__name__})")
    return load_dataset("Capycap-AI/CaptchaSolve30k", split="train", streaming=True)

humans, cnt, skipped = [], Counter(), Counter()
t0 = time.time()
for r in stream():
    g = r.get("gameType")
    if g not in GAMES or cnt[g] >= N_PER_GAME:
        if all(cnt[x] >= N_PER_GAME for x in GAMES): break
        continue
    if r.get("touchscreen") and not INCLUDE_TOUCH: skipped["touch"] += 1; continue
    ti = r.get("tickInputs") or []
    if len(ti) < MIN_TICKS: skipped["short"] += 1; continue
    a = np.array([[d["x"], d["y"], float(bool(d["isDown"]))] for d in ti], dtype=np.float32)
    humans.append(dict(game=g, xy=a)); cnt[g] += 1
print(f"{len(humans)} human sessions {dict(cnt)} in {time.time() - t0:.0f}s; skipped {dict(skipped)}")

In [ ]:
# 4. Task skeletons: the waypoints a session must pass through (presses, releases, turns, every 0.4 s)
def skeleton(a, every=96, turn_deg=50, step=12):
    """Indices of waypoints in a tick array a (T×3)."""
    T = len(a); keep = {0, T - 1}
    keep |= set((np.flatnonzero(np.diff(a[:, 2]) != 0) + 1).tolist())           # press / release: first tick of the new state
    keep |= set(range(0, T, every))
    p = a[::step, :2]
    if len(p) > 2:                                                               # sharp turns
        v = np.diff(p, axis=0); ang = np.arctan2(v[:, 1], v[:, 0])
        dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
        moving = (np.linalg.norm(v[:-1], axis=1) > 0.5) & (np.linalg.norm(v[1:], axis=1) > 0.5)
        keep |= set(((np.flatnonzero((dang > np.radians(turn_deg)) & moving) + 1) * step).tolist())
    return np.array(sorted(i for i in keep if i < T))

for h in humans: h["wp"] = skeleton(h["xy"])

In [ ]:
# 5. Splits of the humans: a fixed replay pool, the rest is split per seed later
idx = np.arange(len(humans))
games_arr = np.array([h["game"] for h in humans])
work_idx, replay_idx = train_test_split(idx, test_size=0.20, stratify=games_arr, random_state=7)
print(f"humans for train/val/test: {len(work_idx)} · replay pool (never trained or tested on): {len(replay_idx)}")

# replay library: every waypoint-to-waypoint segment of the pool humans
LIB, LIB_KEY = [], []
for i in replay_idx:
    a, wp = humans[i]["xy"], humans[i]["wp"]
    for s, e in zip(wp[:-1], wp[1:]):
        seg = a[s:e + 1, :2]
        d = float(np.hypot(*(seg[-1] - seg[0])))
        if d < 2 or len(seg) < 4: continue
        th = math.atan2(seg[-1, 1] - seg[0, 1], seg[-1, 0] - seg[0, 0])
        LIB.append(seg); LIB_KEY.append((d, math.cos(th), math.sin(th)))
LIB_KEY = np.array(LIB_KEY); D_MED = np.median(LIB_KEY[:, 0])
KD = cKDTree(np.c_[LIB_KEY[:, 0] / D_MED, LIB_KEY[:, 1], LIB_KEY[:, 2]])
print(f"replay library: {len(LIB):,} human movement segments")

In [ ]:
# 6. Bot generators — same waypoints and button states as the source human, own movement
def ticks_for(dur): return max(2, int(round(dur / TICK)))

def seg_linear(A, B, speed, noise=0.0):
    d = np.hypot(*(B - A)); n = ticks_for(d / speed)
    t = np.linspace(0, 1, n)[:, None]
    p = A + (B - A) * t
    return p + rng.normal(0, noise, p.shape) if noise else p

def seg_bezier(A, B, tremor=False):
    d = np.hypot(*(B - A))
    if d < 0.5: return np.repeat(A[None], 2, 0)
    dur = (0.10 + 0.11 * math.log2(d / 6 + 1)) * math.exp(rng.normal(0, 0.25))
    n = ticks_for(dur); t = np.linspace(0, 1, n)
    if tremor:                                                   # two-phase sub-movement speed profile
        t = np.where(t < 0.7, 0.85 * (t / 0.7) ** 1.2, 0.85 + 0.15 * ((t - 0.7) / 0.3))
    s = t * t * (3 - 2 * t)
    nrm = np.array([-(B - A)[1], (B - A)[0]]) / d
    bend = rng.uniform(-0.25, 0.25) * d
    C1, C2 = A + (B - A) * 0.3 + nrm * bend, A + (B - A) * 0.7 + nrm * bend * 0.5
    u = (1 - s)[:, None]; s2 = s[:, None]
    p = u**3 * A + 3 * u**2 * s2 * C1 + 3 * u * s2**2 * C2 + s2**3 * B
    if tremor:
        f = rng.uniform(8, 12); amp = rng.uniform(0.15, 0.5); ph = rng.uniform(0, 6.3, 2)
        tt = np.arange(n) * TICK
        p = p + amp * np.c_[np.sin(2 * np.pi * f * tt + ph[0]), np.sin(2 * np.pi * f * tt + ph[1])]
    return p

def seg_bezier_tremor(A, B):
    d = np.hypot(*(B - A))
    if d > 10 and rng.random() < 0.2:                            # overshoot past B, then correct back
        O = B + (B - A) / d * rng.uniform(2, 6) + rng.normal(0, 1, 2)
        return np.vstack([seg_bezier(A, O, tremor=True), seg_linear(O, B, rng.uniform(40, 90))[1:]])
    return seg_bezier(A, B, tremor=True)

def seg_replay(A, B):
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=8)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    v0 = seg[-1] - seg[0]; sc = d / max(np.hypot(*v0), 1e-6)
    rot = th - math.atan2(v0[1], v0[0]); c, s_ = math.cos(rot), math.sin(rot)
    q = (seg - seg[0]) * sc
    return A + np.c_[q[:, 0] * c - q[:, 1] * s_, q[:, 0] * s_ + q[:, 1] * c]

def seg_replay_raw(A, B):
    """Translation only: the nearest human segment by length and direction, NOT rotated or scaled.
    It ends near B rather than on it; the next move starts from wherever it ended."""
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=3)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    return A + (seg - seg[0])

def sensor_profile(a):
    """(update rate per 240 Hz tick while moving, smallest coordinate step) of a session."""
    p = a[:, :2]; ch = np.any(np.diff(p, axis=0) != 0, axis=1)
    near = np.convolve(ch.astype(float), np.ones(9), "same") > 0
    r = float(ch[near].mean()) if near.any() else 1.0
    d = np.abs(np.diff(p, axis=0)).ravel(); d = d[d > 1e-6]
    return min(max(r, 0.05), 1.0), (float(np.percentile(d, 1)) if len(d) else 0.0)

def apply_sensor(b, r, q):
    """Sample-and-hold at rate r (random phase), then snap to a lattice of step q. The button column is untouched."""
    if r < 0.999:
        k = np.arange(len(b)); ph = rng.random()
        upd = np.floor(k * r + ph) > np.floor((k - 1) * r + ph); upd[0] = True
        b[:, :2] = b[np.maximum.accumulate(np.where(upd, k, 0)), :2]
    if q > 1e-4: b[:, :2] = np.round(b[:, :2] / q) * q
    return b

def resample_replay(b, r, q, max_gap=12):
    """Stitched replay → one device: interpolate between position updates (gaps ≤ max_gap ticks = 50 ms;
    longer gaps are real pauses and stay held), then sample-and-hold at rate r and snap to lattice q."""
    p = b[:, :2].astype(np.float64); t = np.arange(len(b))
    ch = np.r_[True, np.any(np.diff(p, axis=0) != 0, axis=1)]; k = np.flatnonzero(ch)
    if len(k) >= 2:
        prev = k[np.searchsorted(k, t, side="right") - 1]
        nxt = k[np.minimum(np.searchsorted(k, t, side="left"), len(k) - 1)]
        smooth = np.c_[np.interp(t, k, p[k, 0]), np.interp(t, k, p[k, 1])]
        p = np.where(((nxt - prev) <= max_gap)[:, None], smooth, p[prev])
    b = b.copy(); b[:, :2] = p
    r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]                 # one correction step, as for L1–L4
    return apply_sensor(b, min(1.0, r * r / max(r1, 1e-3)), q)

def make_bot(h, level):
    a, wp = h["xy"], h["wp"]
    speed = rng.uniform(60, 140)
    out = []; cur = a[0, :2].astype(float)
    for s, e in zip(wp[:-1], wp[1:]):
        A, B, down = a[s, :2].astype(float), a[e, :2].astype(float), a[s, 2]
        if level == "human_replay_raw": A = cur
        if level == "linear":          p = seg_linear(A, B, speed)
        elif level == "linear_jitter": p = seg_linear(A, B, speed * math.exp(rng.normal(0, 0.4)), noise=0.3)
        elif level == "bezier_fitts":  p = seg_bezier(A, B)
        elif level == "bezier_tremor": p = seg_bezier_tremor(A, B)
        elif level == "human_replay":  p = seg_replay(A, B)
        else:                          p = seg_replay_raw(A, B)
        cur = p[-1].astype(float)
        out.append(np.c_[p, np.full(len(p), down)])
        if a[e, 2] != down:                                          # press or release at B: dwell, then switch
            pause = {"linear": 0.08, "linear_jitter": rng.uniform(0.02, 0.25)}.get(level, rng.lognormal(math.log(0.08), 0.6))
            n = ticks_for(pause)
            out.append(np.c_[np.repeat(p[-1:], n, 0), np.full(n, a[e, 2])])
    b = np.vstack(out).astype(np.float32)
    if MATCH_SENSOR:
        if "sensor" not in h: h["sensor"] = sensor_profile(a)
        r, q = h["sensor"]
        if level not in REPLAY:                                    # replay keeps the human update timing of its segments
            r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]   # one correction step: lattice snapping drops updates
            r = min(1.0, r * r / max(r1, 1e-3))
            b = apply_sensor(b, r, q)
        elif REPLAY_RESAMPLE: b = resample_replay(b, r, q)       # replay: one device rate + lattice (see 9c/9d)
    b[:, :2] = np.clip(b[:, :2], 0, 200)
    return b

bots = []
for lv in LEVELS:
    src = rng.choice(work_idx, size=int(BOT_SHARE * len(work_idx)), replace=False)
    for i in src:
        bots.append(dict(game=humans[i]["game"], xy=make_bot(humans[i], lv), level=lv, src=int(i)))
print(f"{len(bots)} bot sessions:", dict(Counter(b["level"] for b in bots)))

In [ ]:
# 7. Features for the three views
def dyn(a):
    p, dn = a[:, :2].astype(np.float64), a[:, 2]
    v = np.diff(p, axis=0) / TICK; sp = np.hypot(v[:, 0], v[:, 1])
    acc = np.diff(sp) / TICK if len(sp) > 1 else np.zeros(1)
    jerk = np.diff(acc) / TICK if len(acc) > 1 else np.zeros(1)
    mv = sp > 1.0
    ang = np.arctan2(v[:, 1], v[:, 0]); dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
    dang = dang[mv[1:] & mv[:-1]] if len(dang) else np.zeros(1)
    plen = sp.sum() * TICK; disp = np.hypot(*(p[-1] - p[0]))
    idle = ~mv; runs = np.diff(np.r_[0, idle.astype(int), 0]); n_pause = int((runs == 1).sum())
    presses = int((np.diff(dn) > 0).sum()); dur = len(a) * TICK
    hist = np.histogram(ang[mv], bins=8, range=(-np.pi, np.pi))[0] + 1e-9; hist = hist / hist.sum()
    # local speed minima per second = sub-movements
    s_sm = np.convolve(sp, np.ones(9) / 9, "same")
    minima = int(((s_sm[1:-1] < s_sm[:-2]) & (s_sm[1:-1] < s_sm[2:]) & (s_sm[1:-1] < np.percentile(s_sm, 60))).sum()) if len(s_sm) > 2 else 0
    # tremor band energy share (8–12 Hz) of the speed signal
    if len(sp) > 64:
        F = np.abs(np.fft.rfft(sp - sp.mean())) ** 2; fr = np.fft.rfftfreq(len(sp), TICK)
        trem = F[(fr >= 8) & (fr <= 12)].sum() / (F[fr > 0.5].sum() + 1e-9)
    else: trem = 0.0
    q = lambda x, k: float(np.percentile(x, k)) if len(x) else 0.0
    return [math.log1p(dur), math.log1p(plen), disp / (plen + 1e-9),
            sp.mean(), sp.std(), q(sp, 10), q(sp, 50), q(sp, 90), sp.max(),
            np.abs(acc).mean(), acc.std(), np.abs(jerk).mean(), jerk.std(),
            dang.mean() if len(dang) else 0, dang.std() if len(dang) else 0,
            idle.mean(), n_pause / dur, dn.mean(), presses / dur,
            float(-(hist * np.log(hist)).sum()), minima / dur, trem,
            float((np.abs(np.diff(p, axis=0)).sum(1) == 0).mean()),       # exactly repeated positions
            np.ptp(p[:, 0]), np.ptp(p[:, 1]),
            float(np.corrcoef(sp[:-1], sp[1:])[0, 1]) if len(sp) > 3 and sp.std() > 0 else 0.0]

STAT_NAMES = ["log_dur", "log_path", "straightness", "speed_mean", "speed_std", "speed_p10", "speed_p50",
              "speed_p90", "speed_max", "acc_mean", "acc_std", "jerk_mean", "jerk_std", "turn_mean", "turn_std",
              "idle_share", "pauses_per_s", "down_share", "presses_per_s", "direction_entropy",
              "submovements_per_s", "tremor_share", "repeat_share", "x_range", "y_range", "speed_autocorr"]

def seq_view(a):
    s = a[::SEQ_STRIDE]
    d = np.diff(s[:, :2], axis=0)
    x = np.c_[d / 5.0, s[1:, 2]].astype(np.float32)[:SEQ_MAX]
    return x if len(x) else np.zeros((1, 3), np.float32)

def graph_view(a):
    s = a[::SEQ_STRIDE, :2]; d = np.diff(s, axis=0); mv = np.hypot(d[:, 0], d[:, 1]) > 0.3
    dirs = ((np.arctan2(d[:, 1], d[:, 0]) + np.pi) / (2 * np.pi) * 8).astype(int) % 8
    dm = dirs[mv]; M = np.ones((8, 8))
    for x, y in zip(dm[:-1], dm[1:]): M[x, y] += 1
    M = M / M.sum(1, keepdims=True)
    cells = (np.clip(s[:, 0], 0, 199.9) // 20 * 10 + np.clip(s[:, 1], 0, 199.9) // 20).astype(int)
    ch = cells[np.r_[True, cells[1:] != cells[:-1]]]
    tr = Counter(zip(ch[:-1], ch[1:])); tot = max(sum(tr.values()), 1)
    pt = np.array(list(tr.values())) / tot if tr else np.ones(1)
    seen, rev = set(), 0
    for c in ch: rev += c in seen; seen.add(c)
    return np.r_[M.ravel(), float(-(pt * np.log(pt)).sum()), rev / max(len(ch), 1), len(seen) / 100.0].astype(np.float32)

GRAPH_NAMES = [f"dir{i}->{j}" for i in range(8) for j in range(8)] + ["cell_transition_entropy", "cell_revisit_ratio", "unique_cells"]
ALL = humans + bots
Y = np.array([0] * len(humans) + [1] * len(bots))
LEVEL = np.array(["human"] * len(humans) + [b["level"] for b in bots])
ORIGIN = np.array(list(range(len(humans))) + [b["src"] for b in bots])     # source human of each bot
t0 = time.time()
X_STAT = np.nan_to_num(np.array([dyn(s["xy"]) for s in ALL], dtype=np.float32))
SEQS = [seq_view(s["xy"]) for s in ALL]
X_GRAPH = np.array([graph_view(s["xy"]) for s in ALL], dtype=np.float32)
print(f"features for {len(ALL)} sessions in {time.time() - t0:.0f}s · stat {X_STAT.shape[1]} · graph {X_GRAPH.shape[1]}")

In [ ]:
# 7b. Generator-artifact check: can a SINGLE feature separate humans from a bot level?
# AUC ≥ 0.95 on one low-level feature means that level is caught by how it was generated, not by behaviour.
sens = np.array([sensor_profile(s["xy"]) for s in ALL])
F = pd.DataFrame(X_STAT, columns=STAT_NAMES).assign(update_rate=sens[:, 0], lattice_step=sens[:, 1])
CHK = ["update_rate", "lattice_step", "repeat_share", "idle_share", "pauses_per_s", "log_dur", "presses_per_s", "down_share"]
display(F[CHK].assign(level=LEVEL).groupby("level").median().round(3))
auc = pd.DataFrame({lv: {c: max(a_ := roc_auc_score(np.r_[np.zeros((LEVEL == "human").sum()), np.ones((LEVEL == lv).sum())],
                                                     np.r_[F[c][LEVEL == "human"], F[c][LEVEL == lv]]), 1 - a_)
                         for c in STAT_NAMES + ["update_rate", "lattice_step"]} for lv in LEVELS})
print("best single-feature AUC per level (human vs level):")
display(pd.DataFrame({"feature": auc.idxmax(), "AUC": auc.max().round(3)}))


In [ ]:
# 8. GS-Fusion for mouse data (stat MLP · LSTM over (dx, dy, button) · transition-graph MLP · softmax gate)
LENS = np.array([len(s) for s in SEQS]); SEQ = np.zeros((len(SEQS), SEQ_MAX, 3), np.float32)
for i, s in enumerate(SEQS): SEQ[i, :len(s)] = s

class GSNet(nn.Module):
    def __init__(self, views, fusion, n_stat, n_graph):
        super().__init__(); self.views, self.fusion = list(views), fusion
        mlp = lambda n: nn.Sequential(nn.Linear(n, 64), nn.ReLU(), nn.Linear(64, D), nn.ReLU())
        if "stat" in views: self.f_stat = mlp(n_stat)
        if "graph" in views: self.f_graph = mlp(n_graph)
        if "seq" in views:
            self.inp = nn.Linear(3, 16); self.lstm = nn.LSTM(16, D, batch_first=True); self.Ws = nn.Linear(D, D)
        k = len(self.views)
        if fusion == "gated": self.gate = nn.Linear(k * D, k)
        self.head = nn.Linear(k * D if fusion == "concat" else D, 1)
    def forward(self, xs, sq, ln, xg):
        h = []
        for v in self.views:
            if v == "stat": h.append(self.f_stat(xs))
            elif v == "graph": h.append(self.f_graph(xg))
            else:
                pk = pack_padded_sequence(torch.relu(self.inp(sq)), ln.cpu(), batch_first=True, enforce_sorted=False)
                _, (hn, _) = self.lstm(pk); h.append(torch.tanh(self.Ws(hn[-1])))
        H = torch.stack(h, 1); g = None
        if self.fusion == "single": z = H[:, 0]
        elif self.fusion == "concat": z = H.flatten(1)
        else: g = torch.softmax(self.gate(H.flatten(1)), -1); z = (g.unsqueeze(-1) * H).sum(1)
        return self.head(z).squeeze(-1), g

def T_(ix, XS, XG):
    return (torch.tensor(XS[ix]), torch.tensor(SEQ[ix]), torch.tensor(LENS[ix]), torch.tensor(XG[ix]),
            torch.tensor(Y[ix], dtype=torch.float32))

@torch.no_grad()
def predict(m, d, bs=1024):
    m.eval(); P, G = [], []
    for i in range(0, len(d[0]), bs):
        lo, g = m(d[0][i:i+bs].to(DEVICE), d[1][i:i+bs].to(DEVICE), d[2][i:i+bs], d[3][i:i+bs].to(DEVICE))
        P.append(torch.sigmoid(lo).cpu())
        if g is not None: G.append(g.cpu())
    return torch.cat(P).numpy(), (torch.cat(G).numpy() if G else None)

def train_net(views, fusion, lam, tr, va, seed):
    torch.manual_seed(seed)
    m = GSNet(views, fusion, X_STAT.shape[1], X_GRAPH.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR); y = tr[4]
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((y == 0).sum() / max((y == 1).sum(), 1)), device=DEVICE))
    best, state, wait = -1, None, 0
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(y))
        for i in range(0, len(y), BATCH):
            b = perm[i:i+BATCH]
            lo, g = m(tr[0][b].to(DEVICE), tr[1][b].to(DEVICE), tr[2][b], tr[3][b].to(DEVICE))
            loss = bce(lo, y[b].to(DEVICE))
            if g is not None and lam > 0:
                k = g.shape[1]; loss = loss + lam * k * ((g.mean(0) - 1 / k) ** 2).sum()
            opt.zero_grad(); loss.backward(); opt.step()
        ap = average_precision_score(va[4].numpy(), predict(m, va)[0])
        if ap > best + 1e-5: best, wait, state = ap, 0, copy.deepcopy(m.state_dict())
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(state); return m

In [ ]:
# 9. Experiments E1 (all levels known) and E2 (L5 unseen in training)
def fpr_threshold(y, s):
    h = np.sort(s[y == 0]); k = int(np.floor(TARGET_FPR * len(h)))
    return float(np.nextafter(h[len(h) - k - 1], np.inf))

VARIANTS = {"GS: stat": (["stat"], "single", 0), "GS: seq": (["seq"], "single", 0), "GS: graph": (["graph"], "single", 0),
            "GS: concat": (["stat", "seq", "graph"], "concat", 0), "GS-Fusion (gated)": (["stat", "seq", "graph"], "gated", LAMBDA_BAL)}
if SKIP_SEQ is None: SKIP_SEQ = DEVICE == "cpu"
if SKIP_SEQ:
    VARIANTS = {k: v for k, v in VARIANTS.items() if "seq" not in v[0]}
    print("no GPU → running without the sequence (LSTM) models:", list(VARIANTS))
# Results are saved after every (experiment, seed); re-running this cell skips finished ones,
# so a Colab disconnect or a stopped cell loses at most one seed. Delete PROGRESS to start over.
import pickle
PROGRESS = f"{OUT}/progress_v5.pkl"                  # v5 generator (replay re-sampled, rate-calibrated); older progress files are ignored
if os.path.exists(PROGRESS):
    rows, gate_rows, imp_rows, eff_rows, done = pickle.load(open(PROGRESS, "rb"))
    print(f"resuming: {len(done)} (experiment, seed) runs already finished")
else:
    rows, gate_rows, imp_rows, eff_rows, done = [], [], [], [], set()
for exp in ["E1 known attackers", "E2 L5 unseen"]:
    for seed in SEEDS:
        if (exp, seed) in done: continue
        t_run = time.time()
        # split by SOURCE human, so a human and the bots built on its skeleton stay on one side
        rest, te_h = train_test_split(work_idx, test_size=0.2, stratify=games_arr[work_idx], random_state=seed)
        tr_h, va_h = train_test_split(rest, test_size=0.2, stratify=games_arr[rest], random_state=seed)
        side = np.full(len(ALL), -1)
        for k, part in enumerate([tr_h, va_h, te_h]): side[np.isin(ORIGIN, part)] = k
        tr_i, va_i, te_i = (np.flatnonzero(side == k) for k in range(3))
        if exp.startswith("E2"):
            tr_i = tr_i[~np.isin(LEVEL[tr_i], REPLAY)]; va_i = va_i[~np.isin(LEVEL[va_i], REPLAY)]
        ss, sg = StandardScaler().fit(X_STAT[tr_i]), StandardScaler().fit(X_GRAPH[tr_i])
        XS, XG = ss.transform(X_STAT).astype(np.float32), sg.transform(X_GRAPH).astype(np.float32)
        yv, yt, lt = Y[va_i], Y[te_i], LEVEL[te_i]

        def record_pred(name, pred, score):
            row = dict(experiment=exp, seed=seed, model=name, ROC_AUC=roc_auc_score(yt, score),
                       FPR=float(pred[yt == 0].mean()), recall_all=float(pred[yt == 1].mean()))
            for lv in LEVELS: row[lv] = float(pred[lt == lv].mean())
            rows.append(row)

        def record(name, s_va, s_te):
            record_pred(name, s_te >= fpr_threshold(yv, s_va), s_te)
            SCORES[name] = (s_va, s_te)

        def classic(name, model, X):
            m_ = model.fit(X[tr_i], Y[tr_i]); record(name, m_.predict_proba(X[va_i])[:, 1], m_.predict_proba(X[te_i])[:, 1])
            return m_

        def any_view(name, views):
            """Each view is its own detector; flag if ANY view flags, with the FPR budget split evenly (Bonferroni).
            Per view: p = share of validation humans scoring at least as high."""
            pv = []
            for v in views:
                s_va, s_te = SCORES[v]; h = np.sort(s_va[yv == 0])
                pv.append((len(h) - np.searchsorted(h, s_te, side="left") + 1) / (len(h) + 1))
            p_min = np.min(pv, axis=0)
            record_pred(name, p_min <= TARGET_FPR / len(views), -np.log(p_min))

        SCORES = {}
        mk_rf = lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=seed)
        mk_lgb = lambda: lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, class_weight="balanced",
                                            random_state=seed, verbose=-1)

        lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(XS[tr_i], Y[tr_i])
        record("Logistic regression (stat)", lr.predict_proba(XS[va_i])[:, 1], lr.predict_proba(XS[te_i])[:, 1])
        XSG = np.hstack([X_STAT, X_GRAPH])
        classic("Random Forest (stat)", mk_rf(), X_STAT)
        classic("Random Forest (stat+graph)", mk_rf(), XSG)
        rf_g = classic("Random Forest (graph only)", mk_rf(), X_GRAPH)                  # control: features or model?
        classic("Logistic regression (graph only)", LogisticRegression(max_iter=3000, class_weight="balanced"), XG)
        classic("LightGBM (stat)", mk_lgb(), X_STAT)
        classic("LightGBM (stat+graph)", mk_lgb(), XSG)
        classic("LightGBM (graph only)", mk_lgb(), X_GRAPH)
        if exp.startswith("E2"):
            imp_rows.append(pd.Series(rf_g.feature_importances_, GRAPH_NAMES, name=seed))
            for lv in REPLAY:                                        # which graph features separate humans from replay?
                h, b = X_GRAPH[te_i][lt == "human"], X_GRAPH[te_i][lt == lv]
                d_ = (b.mean(0) - h.mean(0)) / np.sqrt((b.var(0) + h.var(0)) / 2 + 1e-9)
                eff_rows.append(pd.Series(d_, GRAPH_NAMES, name=f"{lv}|{seed}"))
        tr, va, te = T_(tr_i, XS, XG), T_(va_i, XS, XG), T_(te_i, XS, XG)
        for name, (views, fusion, lam) in VARIANTS.items():
            m = train_net(views, fusion, lam, tr, va, seed)
            s_va, _ = predict(m, va); s_te, g = predict(m, te)
            record(name, s_va, s_te)
            if g is None: SCORES[name] = (s_va, s_te)
            if g is not None:
                for lv in ["human"] + LEVELS:
                    if (lt == lv).any():
                        gm = g[lt == lv].mean(0); gate_rows.append(dict(experiment=exp, seed=seed, profile=lv,
                                                                        g_stat=gm[0], g_seq=gm[1], g_graph=gm[2]))
        if "GS: seq" in SCORES: any_view("Any view flags (stat | seq | graph)", ["GS: stat", "GS: seq", "GS: graph"])
        any_view("Any view flags (stat | graph)", ["GS: stat", "GS: graph"])
        r = pd.DataFrame(rows); last = r[(r.experiment == exp) & (r.seed == seed)].set_index("model")
        done.add((exp, seed))
        pickle.dump((rows, gate_rows, imp_rows, eff_rows, done), open(PROGRESS, "wb"))
        print(f"{exp} | seed {seed} | {time.time() - t_run:.0f}s | replay recall at FPR≤{TARGET_FPR:.0%} (rotated/scaled · raw):")
        for k in last.index: print(f"    {k:<38} {last.loc[k, 'human_replay']:.2f} · {last.loc[k, 'human_replay_raw']:.2f}")

In [ ]:
# 9b. What does RF (stat) use to catch UNSEEN replay? Same E2 splits, all seeds, no neural training (≈1 min)
imp, perm, single = [], [], []
for seed in SEEDS:
    rest, te_h = train_test_split(work_idx, test_size=0.2, stratify=games_arr[work_idx], random_state=seed)
    tr_h, va_h = train_test_split(rest, test_size=0.2, stratify=games_arr[rest], random_state=seed)
    tr_i = np.flatnonzero(np.isin(ORIGIN, tr_h) & ~np.isin(LEVEL, REPLAY))      # L1–L4 + humans only
    te_i = np.flatnonzero(np.isin(ORIGIN, te_h) & np.isin(LEVEL, ["human"] + REPLAY))
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=seed).fit(X_STAT[tr_i], Y[tr_i])
    Xt, yt = X_STAT[te_i], Y[te_i]
    base = roc_auc_score(yt, rf.predict_proba(Xt)[:, 1])
    rs = np.random.default_rng(seed); d = {}
    for j, c in enumerate(STAT_NAMES):                                          # permutation: AUC drop human vs replay
        Xp = Xt.copy(); Xp[:, j] = rs.permutation(Xp[:, j])
        d[c] = base - roc_auc_score(yt, rf.predict_proba(Xp)[:, 1])
    perm.append(pd.Series(d)); imp.append(pd.Series(rf.feature_importances_, STAT_NAMES))
    single.append(pd.Series({c: roc_auc_score(yt, Xt[:, j]) for j, c in enumerate(STAT_NAMES)}))  # model-free
    print(f"seed {seed}: RF(stat) ROC-AUC human vs replay = {base:.3f}")

T = pd.DataFrame({"perm_AUC_drop": pd.concat(perm, axis=1).mean(1),
                  "RF_impurity": pd.concat(imp, axis=1).mean(1),
                  "single_AUC": pd.concat(single, axis=1).mean(1)})
T["single_sep"] = (T.single_AUC - 0.5).abs() + 0.5                             # direction-free separability
T["replay_higher"] = T.single_AUC > 0.5
print("Features that let RF (trained without replay) catch replay — top 10 by permutation drop:")
display(T.sort_values("perm_AUC_drop", ascending=False).head(10).round(3))
print("\nMost separable single features (human vs replay, no model):")
display(T.sort_values("single_sep", ascending=False).head(8)[["single_AUC", "replay_higher"]].round(3))
T.round(4).to_csv(f"{OUT}/replay_feature_diagnostics.csv")


In [ ]:
# 9c. Where does the replay turning signal come from? joins vs lattice snapping vs the segments themselves
def turns(a):
    v = np.diff(a[:, :2].astype(np.float64), axis=0) / TICK; mv = np.hypot(v[:, 0], v[:, 1]) > 1.0
    dang = np.abs((np.diff(np.arctan2(v[:, 1], v[:, 0])) + np.pi) % (2 * np.pi) - np.pi)
    return dang, mv[1:] & mv[:-1]                                   # dang[k] is the turn at tick k+1

def replay_j(h, level, mode):
    """make_bot for the replay levels, also returning the tick index of every join. mode: raw | snap | resample"""
    a, wp = h["xy"], h["wp"]; out, joins, n = [], [], 0; cur = a[0, :2].astype(float)
    for s, e in zip(wp[:-1], wp[1:]):
        A, B, down = a[s, :2].astype(float), a[e, :2].astype(float), a[s, 2]
        if level == "human_replay_raw": A = cur
        p = seg_replay(A, B) if level == "human_replay" else seg_replay_raw(A, B)
        cur = p[-1].astype(float); out.append(np.c_[p, np.full(len(p), down)]); n += len(p); joins.append(n)
        if a[e, 2] != down:
            k = ticks_for(rng.lognormal(math.log(0.08), 0.6)); out.append(np.c_[np.repeat(p[-1:], k, 0), np.full(k, a[e, 2])]); n += k
    b = np.vstack(out).astype(np.float32)
    r, q = sensor_profile(a)
    if mode == "snap": b = apply_sensor(b, 1.0, q)
    elif mode == "resample": b = resample_replay(b, r, q)
    b[:, :2] = np.clip(b[:, :2], 0, 200)
    return b, joins[:-1]

def turn_stats(a, joins, w=4):
    dang, ok = turns(a); near = np.zeros(len(dang), bool)
    # sessions without two consecutive moving ticks have no defined turn: kept as NaN here; dyn() sets turn_mean = 0
    for j in joins: near[max(0, j - 1 - w): j - 1 + w] = True
    f = lambda m: float(dang[m].mean()) if m.any() else np.nan
    return f(ok), f(ok & ~near), f(ok & near)

S = rng.choice(work_idx, size=min(400, len(work_idx)), replace=False)
H = np.array([turn_stats(humans[i]["xy"], humans[i]["wp"][1:-1]) for i in S])  # humans: same waypoints masked
rows_9c = []
for mode in ["raw", "snap", "resample"]:
    for lv in REPLAY:
        BB = [replay_j(humans[i], lv, mode)[0] for i in S]
        B = np.array([turn_stats(*replay_j(humans[i], lv, mode)) for i in S])
        ur = lambda arr: roc_auc_score(np.r_[np.zeros(len(S)), np.ones(len(S))], arr)
        auc = lambda c: roc_auc_score(np.r_[np.zeros(len(S)), np.ones(len(S))], np.nan_to_num(np.r_[H[:, c], B[:, c]]))
        rows_9c.append(dict(level=lv, mode=mode, AUC_turn=auc(0), AUC_away_from_joins=auc(1), AUC_at_joins=auc(2),
                            AUC_update_rate=ur(np.r_[[sensor_profile(humans[i]["xy"])[0] for i in S], [sensor_profile(x)[0] for x in BB]]),
                            human_turn_undefined=np.isnan(H[:, 0]).mean(), bot_turn_undefined=np.isnan(B[:, 0]).mean(),
                            human_turn_median=np.nanmedian(H[:, 0]), bot_turn_median=np.nanmedian(B[:, 0])))
t9c = pd.DataFrame(rows_9c).round(3); display(t9c); t9c.to_csv(f"{OUT}/replay_turn_joins.csv", index=False)


In [ ]:
# 9d. Control: self-replay (a human's OWN other segments) vs cross-human replay (the pool library), no snapping
def self_replay(h, level, resample=False):
    a, wp = h["xy"], h["wp"]; segs, keys, sid = [], [], []
    for k, (s, e) in enumerate(zip(wp[:-1], wp[1:])):
        seg = a[s:e + 1, :2].astype(float); d = float(np.hypot(*(seg[-1] - seg[0])))
        if d < 2 or len(seg) < 4: continue
        th = math.atan2(seg[-1, 1] - seg[0, 1], seg[-1, 0] - seg[0, 0])
        segs.append(seg); keys.append((d, math.cos(th), math.sin(th))); sid.append(k)
    if len(segs) < 10: return None
    keys, sid = np.array(keys), np.array(sid); dm = np.median(keys[:, 0])
    kd = cKDTree(np.c_[keys[:, 0] / dm, keys[:, 1], keys[:, 2]])
    out, joins, n = [], [], 0; cur = a[0, :2].astype(float)
    for k, (s, e) in enumerate(zip(wp[:-1], wp[1:])):
        A, B, down = a[s, :2].astype(float), a[e, :2].astype(float), a[s, 2]
        if level == "human_replay_raw": A = cur
        d = float(np.hypot(*(B - A)))
        if d < 2: p = seg_linear(A, B, 60)
        else:
            th = math.atan2(B[1] - A[1], B[0] - A[0])
            _, c = kd.query([d / dm, math.cos(th), math.sin(th)], k=min(9, len(segs)))
            c = [x for x in np.atleast_1d(c) if sid[x] != k]               # never its own segment
            seg = segs[int(rng.choice(c))]
            if level == "human_replay":
                v0 = seg[-1] - seg[0]; sc = d / max(np.hypot(*v0), 1e-6); rot = th - math.atan2(v0[1], v0[0])
                q = (seg - seg[0]) * sc; cs, sn = math.cos(rot), math.sin(rot)
                p = A + np.c_[q[:, 0] * cs - q[:, 1] * sn, q[:, 0] * sn + q[:, 1] * cs]
            else: p = A + (seg - seg[0])
        cur = p[-1].astype(float); out.append(np.c_[p, np.full(len(p), down)]); n += len(p); joins.append(n)
        if a[e, 2] != down:
            kk = ticks_for(rng.lognormal(math.log(0.08), 0.6)); out.append(np.c_[np.repeat(p[-1:], kk, 0), np.full(kk, a[e, 2])]); n += kk
    b = np.vstack(out).astype(np.float32)
    if resample: b = resample_replay(b, *sensor_profile(a))
    b[:, :2] = np.clip(b[:, :2], 0, 200)
    return b, joins[:-1]

rows_9d = []
for lv in REPLAY:
    for src, fn in [("cross-human (pool)", lambda h, l: replay_j(h, l, "raw")), ("cross-human, resampled", lambda h, l: replay_j(h, l, "resample")), ("self (own segments)", self_replay), ("self, resampled", lambda h, l: self_replay(h, l, True))]:
        Hs, Bs = [], []
        for i in S:
            r_ = fn(humans[i], lv)
            if r_ is None: continue
            Hs.append(turn_stats(humans[i]["xy"], humans[i]["wp"][1:-1])); Bs.append(turn_stats(*r_))
        Hs, Bs = np.array(Hs), np.array(Bs); yy = np.r_[np.zeros(len(Hs)), np.ones(len(Bs))]
        auc = lambda c: roc_auc_score(yy, np.nan_to_num(np.r_[Hs[:, c], Bs[:, c]]))
        rows_9d.append(dict(level=lv, library=src, n=len(Hs), AUC_all=auc(0), AUC_away_from_joins=auc(1),
                            human_median=np.nanmedian(Hs[:, 0]), bot_median=np.nanmedian(Bs[:, 0]),
                            human_undefined=np.isnan(Hs[:, 0]).mean(), bot_undefined=np.isnan(Bs[:, 0]).mean()))
t9d = pd.DataFrame(rows_9d).round(3); display(t9d); t9d.to_csv(f"{OUT}/replay_self_vs_cross.csv", index=False)


In [ ]:
# 9e. The remaining replay signal: sub-movements (speed minima). At the joins (stitching) or everywhere (generator)?
def minima_mask(a):
    v = np.diff(a[:, :2].astype(np.float64), axis=0) / TICK; sp = np.hypot(v[:, 0], v[:, 1])
    s = np.convolve(sp, np.ones(9) / 9, "same"); m = np.zeros(len(s), bool)
    if len(s) > 2: m[1:-1] = (s[1:-1] < s[:-2]) & (s[1:-1] < s[2:]) & (s[1:-1] < np.percentile(s, 60))
    return m                                                        # same definition as submovements_per_s in dyn()

def minima_stats(a, joins, w=12):
    m = minima_mask(a); near = np.zeros(len(m), bool)
    for j in joins: near[max(0, j - w): j + w] = True
    return (m.sum() / (len(a) * TICK),                              # per second, whole session
            m[~near].sum() / max((~near).sum() * TICK, 1e-9),       # per second, away from joins
            m[near].sum() / max(len(joins), 1))                     # per join (humans: per waypoint)

Hm = np.array([minima_stats(humans[i]["xy"], humans[i]["wp"][1:-1]) for i in S])
gens = [("cross-human, resampled (= E2 bots)", lambda h, l: replay_j(h, l, "resample")),
        ("self, resampled", lambda h, l: self_replay(h, l, True))]
rows_9e = []
for lv in REPLAY:
    for name, fn in gens:
        pairs = []
        for k, i in enumerate(S):
            r_ = fn(humans[i], lv)
            if r_ is None: continue                                 # self-replay needs ≥ 10 own segments
            pairs.append((Hm[k], minima_stats(*r_), sensor_profile(humans[i]["xy"])[0], sensor_profile(r_[0])[0]))
        Hh, Bb = np.array([p[0] for p in pairs]), np.array([p[1] for p in pairs]); yy = np.r_[np.zeros(len(Hh)), np.ones(len(Bb))]
        uh, ub = [p[2] for p in pairs], [p[3] for p in pairs]
        auc = lambda c: roc_auc_score(yy, np.r_[Hh[:, c], Bb[:, c]])
        rows_9e.append(dict(level=lv, generator=name, AUC_submov_all=auc(0), AUC_submov_away=auc(1), AUC_minima_per_join=auc(2),
                            human_per_s=np.median(Hh[:, 0]), bot_per_s=np.median(Bb[:, 0]),
                            human_per_join=np.median(Hh[:, 2]), bot_per_join=np.median(Bb[:, 2]),
                            AUC_update_rate=roc_auc_score(yy, np.r_[uh, ub]), n=len(pairs)))
t9e = pd.DataFrame(rows_9e).round(3); display(t9e); t9e.to_csv(f"{OUT}/replay_submovements.csv", index=False)


In [ ]:
# 9f. Deep baselines on raw trajectories: dilated 1D-CNN and Transformer (same splits, thresholds, E1/E2 as §9)
# Answers the reviewer question "maybe the models are just weak". Runs only on GPU; saves after every (experiment, seed).
import pickle, time
DEEP_PROGRESS = f"{OUT}/progress_deep.pkl"
deep_rows, deep_done = pickle.load(open(DEEP_PROGRESS, "rb")) if os.path.exists(DEEP_PROGRESS) else ([], set())

# inputs: (dx, dy, button) at 30 Hz from §8 plus speed and log-speed; padded to SEQ_MAX with a mask
_sp = np.hypot(SEQ[..., 0], SEQ[..., 1])
XDEEP = np.concatenate([SEQ, _sp[..., None], np.log1p(_sp)[..., None]], -1).astype(np.float32)
MASK = np.arange(SEQ.shape[1])[None, :] < LENS[:, None]; MASK[:, 0] = True   # never an all-padding row

class CNN1D(nn.Module):
    def __init__(self, c_in, w=64):
        super().__init__(); self.inp = nn.Conv1d(c_in, w, 1)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Conv1d(w, w, 5, padding=2 * d, dilation=d), nn.BatchNorm1d(w), nn.GELU(),
                                                   nn.Conv1d(w, w, 5, padding=2 * d, dilation=d), nn.BatchNorm1d(w)) for d in (1, 2, 4, 8)])
        self.head = nn.Sequential(nn.Linear(2 * w, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x, m):
        h = self.inp(x.transpose(1, 2)); mk = m[:, None, :].float()
        for b in self.blocks: h = torch.nn.functional.gelu(h + b(h)) * mk
        mean = h.sum(-1) / mk.sum(-1).clamp(min=1); mx = h.masked_fill(mk == 0, -1e4).max(-1).values
        return self.head(torch.cat([mean, mx], 1)).squeeze(-1)

class TinyTransformer(nn.Module):
    def __init__(self, c_in, w=64, layers=3, heads=4, max_len=SEQ.shape[1]):
        super().__init__(); self.inp = nn.Linear(c_in, w); self.pos = nn.Parameter(torch.zeros(1, max_len, w))
        enc = nn.TransformerEncoderLayer(w, heads, 2 * w, dropout=0.1, batch_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(enc, layers); self.head = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, 1))
    def forward(self, x, m):
        h = self.enc(self.inp(x) + self.pos[:, :x.shape[1]], src_key_padding_mask=~m)
        mk = m[..., None].float(); return self.head((h * mk).sum(1) / mk.sum(1).clamp(min=1)).squeeze(-1)

@torch.no_grad()
def deep_predict(net, X, M, ix, bs=1024):
    net.eval(); out = []
    for i in range(0, len(ix), bs):
        b = ix[i:i + bs]; out.append(torch.sigmoid(net(torch.tensor(X[b], device=DEVICE), torch.tensor(M[b], device=DEVICE))).cpu())
    return torch.cat(out).numpy()

def deep_train(make, X, M, tr, va, seed, epochs=EPOCHS, patience=PATIENCE):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make().to(DEVICE); opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    pw = float((Y[tr] == 0).sum() / max((Y[tr] == 1).sum(), 1))
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=DEVICE)); best, state, wait = -1, None, 0
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(tr)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            lo = net(torch.tensor(X[b], device=DEVICE), torch.tensor(M[b], device=DEVICE))
            loss = bce(lo, torch.tensor(Y[b], dtype=torch.float32, device=DEVICE))
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        ap = average_precision_score(Y[va], deep_predict(net, X, M, va))
        if ap > best + 1e-5: best, wait, state = ap, 0, copy.deepcopy(net.state_dict())
        else:
            wait += 1
            if wait >= patience: break
    net.load_state_dict(state); return net

DEEP = {"1D-CNN (raw trajectory)": lambda: CNN1D(XDEEP.shape[-1]),
        "Transformer (raw trajectory)": lambda: TinyTransformer(XDEEP.shape[-1])}
if DEVICE == "cpu": print("no GPU: skipping deep baselines"); DEEP = {}
for exp in ["E1 known attackers", "E2 L5 unseen"]:
    for seed in SEEDS:
        if (exp, seed) in deep_done or not DEEP: continue
        t0 = time.time()
        rest, te_h = train_test_split(work_idx, test_size=0.2, stratify=games_arr[work_idx], random_state=seed)   # identical to §9
        tr_h, va_h = train_test_split(rest, test_size=0.2, stratify=games_arr[rest], random_state=seed)
        side = np.full(len(ALL), -1)
        for k, part in enumerate([tr_h, va_h, te_h]): side[np.isin(ORIGIN, part)] = k
        tr_i, va_i, te_i = (np.flatnonzero(side == k) for k in range(3))
        if exp.startswith("E2"):
            tr_i = tr_i[~np.isin(LEVEL[tr_i], REPLAY)]; va_i = va_i[~np.isin(LEVEL[va_i], REPLAY)]
        mu = XDEEP[tr_i][MASK[tr_i]].mean(0); sd = XDEEP[tr_i][MASK[tr_i]].std(0) + 1e-6       # train-only scaling
        Xn = ((XDEEP - mu) / sd * MASK[..., None]).astype(np.float32)
        yv, yt, lt = Y[va_i], Y[te_i], LEVEL[te_i]
        for name, make in DEEP.items():
            net = deep_train(make, Xn, MASK, tr_i, va_i, seed)
            s_va, s_te = deep_predict(net, Xn, MASK, va_i), deep_predict(net, Xn, MASK, te_i)
            pred = s_te >= fpr_threshold(yv, s_va)
            row = dict(experiment=exp, seed=seed, model=name, ROC_AUC=roc_auc_score(yt, s_te),
                       FPR=float(pred[yt == 0].mean()), recall_all=float(pred[yt == 1].mean()))
            for lv in LEVELS: row[lv] = float(pred[lt == lv].mean())
            deep_rows.append(row)
        deep_done.add((exp, seed)); pickle.dump((deep_rows, deep_done), open(DEEP_PROGRESS, "wb"))
        print(f"{exp} seed {seed}: {time.time() - t0:.0f} s")

if deep_rows:
    DEEP_DF = pd.DataFrame(deep_rows); DEEP_DF.to_csv(f"{OUT}/deep_baseline_raw.csv", index=False)
    print(DEEP_DF.groupby(["model", "experiment"])[["human_replay", "human_replay_raw", "FPR", "ROC_AUC"]].mean().round(3))


In [ ]:
# 10. Results
res = pd.DataFrame(rows); gates = pd.DataFrame(gate_rows)
cols = ["ROC_AUC", "FPR", "recall_all"] + LEVELS
tab = res.groupby(["experiment", "model"], sort=False)[cols].agg(["mean", "std"]).round(3)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
for exp in res.experiment.unique():
    print(f"\n{exp} — recall per bot level at FPR ≤ {TARGET_FPR:.0%} (mean over seeds)")
    display(res[res.experiment == exp].groupby("model", sort=False)[cols].mean().round(3))
if len(gates): display(gates.groupby(["experiment", "profile"])[["g_stat", "g_seq", "g_graph"]].mean().round(3))
res.to_csv(f"{OUT}/captcha_raw.csv", index=False); tab.to_csv(f"{OUT}/captcha_table.csv")

# Which graph features carry the replay signal? (E2, RF-on-graph importance and effect size human → replay)
imp = pd.concat(imp_rows, axis=1).mean(1).sort_values(ascending=False)
eff = pd.concat(eff_rows, axis=1)
eff_mean = pd.DataFrame({lv: eff[[c for c in eff.columns if c.startswith(lv + "|")]].mean(1) for lv in REPLAY})
top = eff_mean.reindex(eff_mean.abs().max(1).sort_values(ascending=False).index).head(12).round(2)
print("Graph features that differ most between humans and replay bots (Cohen's d, test sets of E2):"); display(top)
print("Random Forest (graph only) — top-10 importances:"); display(imp.head(10).round(3))
grp = pd.Series({"direction transitions (64)": imp[[n for n in GRAPH_NAMES if n.startswith("dir")]].sum(),
                 "screen-cell transitions (3)": imp[GRAPH_NAMES[64:]].sum()})
print("importance by feature group:"); display(grp.round(3))
top.to_csv(f"{OUT}/graph_effect_sizes.csv"); imp.to_csv(f"{OUT}/graph_importance.csv")
gates.to_csv(f"{OUT}/captcha_gates.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(17, 5), sharey=True)
for a_, exp in zip(ax, res.experiment.unique()):
    m = res[res.experiment == exp].groupby("model", sort=False)[LEVELS].mean()
    m.T.plot(kind="bar", ax=a_, width=0.85, legend=(a_ is ax[0]))
    a_.set_title(exp); a_.set_ylabel(f"recall at FPR ≤ {TARGET_FPR:.0%}"); a_.set_ylim(0, 1.02); a_.tick_params(axis="x", rotation=20)
ax[0].legend(fontsize=6, loc="lower left", ncol=2); plt.tight_layout(); plt.savefig(f"{OUT}/fig_captcha_levels.png", dpi=200); plt.show()

import zipfile
date_files()
with zipfile.ZipFile(f"{BASE}/captcha_results_{RUN_DATE}.zip", "w") as z:
    for p in os.listdir(OUT): z.write(f"{OUT}/{p}", p)
try:
    from google.colab import files; files.download(f"{BASE}/captcha_results_{RUN_DATE}.zip")
except Exception: print(f"saved {BASE}/captcha_results_{RUN_DATE}.zip")